# 数据分析

这份 notebook 用来把当前最小版 ADL 第一轮流程的结果，整理成一份适合机器学习实验汇报的中文分析模板。

它围绕 4 个核心问题组织：

1. 数据长什么样？
2. 模型训练得顺不顺？
3. 模型预测得准不准？
4. 模型为什么这样预测？

这份模板同时支持两条主线：

- `ADL 主线`：直接读取仓库约定的 `delta_dataset.npz`、`training_summary.json`、`uncertainty_latest.json`、`round_001_selection_summary.json`
- `可选扩展主线`：如果你在服务器端额外准备了特征表、逐 epoch 训练 history、逐样本预测结果、可解释模型文件，就能补齐更完整的机器学习图表

适用范围说明：

- 默认按 `回归任务` 设计，主目标通常是 `delta_E`
- 当前仓库并没有跟踪真实数据、模型和结果文件，所以如果本地目录是空壳，下面很多模块会提示“缺少文件并跳过”
- 回到服务器后，只要把本 notebook 放在仓库里并保证路径设置正确，就能直接复用


In [ ]:
# 这一个代码块负责导入常用库、设置显示样式，并给后面的分析提供统一环境。
from __future__ import annotations

import importlib.util
import json
import math
import pickle
import warnings
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from IPython.display import Markdown, display
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


class SimpleSeabornFallback:
    # 当环境里没有 seaborn 时，用一个轻量回退层保证 notebook 还能继续运行。
    def set_theme(self, style: str = "whitegrid", context: str = "talk") -> None:  # noqa: ARG002
        if "seaborn-v0_8-whitegrid" in plt.style.available:
            plt.style.use("seaborn-v0_8-whitegrid")
        else:
            plt.style.use("default")

    def histplot(self, data, kde: bool = False, ax=None, color=None, bins: int = 30, **kwargs):  # noqa: ANN001, ARG002
        ax = ax or plt.gca()
        series = pd.Series(data).dropna()
        if series.empty:
            return ax
        ax.hist(series, bins=bins, color=color, alpha=0.75)
        if kde and len(series) > 1:
            series.plot(kind="density", ax=ax, color=color or "black")
        return ax

    def boxplot(self, data=None, orient: str = "v", ax=None, **kwargs):  # noqa: ANN001, ARG002
        ax = ax or plt.gca()
        if isinstance(data, pd.DataFrame):
            labels = list(data.columns)
            values = [pd.Series(data[column]).dropna().tolist() for column in labels]
            ax.boxplot(values, vert=(orient != "h"), labels=labels)
        return ax

    def heatmap(self, data, cbar: bool = True, yticklabels=True, cmap=None, annot: bool = False, square: bool = False, ax=None, **kwargs):  # noqa: ANN001, ARG002
        ax = ax or plt.gca()
        matrix = np.asarray(data, dtype=float)
        image = ax.imshow(matrix, aspect="auto", cmap=cmap or "viridis")
        if cbar:
            plt.colorbar(image, ax=ax)
        if hasattr(data, "columns"):
            ax.set_xticks(range(len(data.columns)))
            ax.set_xticklabels(list(data.columns), rotation=90)
        if hasattr(data, "index") and yticklabels:
            ax.set_yticks(range(len(data.index)))
            ax.set_yticklabels(list(data.index))
        elif not yticklabels:
            ax.set_yticks([])
        if annot and matrix.size <= 400:
            for row_idx in range(matrix.shape[0]):
                for col_idx in range(matrix.shape[1]):
                    ax.text(col_idx, row_idx, f"{matrix[row_idx, col_idx]:.2f}", ha="center", va="center", fontsize=8)
        if square:
            ax.set_aspect("equal")
        return ax

    def barplot(self, data=None, x=None, y=None, hue=None, ax=None, color=None, **kwargs):  # noqa: ANN001, ARG002
        ax = ax or plt.gca()
        if data is not None and hue:
            pivot = data.pivot_table(index=x, columns=hue, values=y, aggfunc="mean", fill_value=0)
            pivot.plot(kind="bar", ax=ax)
            return ax

        if data is not None:
            x_values = data[x] if isinstance(x, str) else x
            y_values = data[y] if isinstance(y, str) else y
        else:
            x_values = x
            y_values = y

        x_series = pd.Series(x_values)
        y_series = pd.Series(y_values)
        if pd.api.types.is_numeric_dtype(x_series) and not pd.api.types.is_numeric_dtype(y_series):
            ax.barh(list(y_series), list(x_series), color=color)
        else:
            ax.bar(list(x_series), list(y_series), color=color)
        return ax

    def scatterplot(self, data=None, x=None, y=None, ax=None, alpha: float = 1.0, s: float = 40, label=None, color=None, **kwargs):  # noqa: ANN001, ARG002
        ax = ax or plt.gca()
        if data is not None:
            x_values = data[x] if isinstance(x, str) else x
            y_values = data[y] if isinstance(y, str) else y
        else:
            x_values = x
            y_values = y
        ax.scatter(x_values, y_values, alpha=alpha, s=s, label=label, color=color)
        return ax


try:
    import seaborn as sns
    HAS_SEABORN = True
except ModuleNotFoundError:
    sns = SimpleSeabornFallback()
    HAS_SEABORN = False
    print("提示：当前环境未安装 seaborn，下面会自动回退到 matplotlib 风格，图能继续画，但样式会更朴素。")

warnings.filterwarnings("ignore")
pd.options.display.max_columns = 200
pd.options.display.max_rows = 200

plt.rcParams["figure.dpi"] = 120
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["font.sans-serif"] = [
    "Microsoft YaHei",
    "SimHei",
    "Noto Sans CJK SC",
    "Arial Unicode MS",
    "DejaVu Sans",
]
sns.set_theme(style="whitegrid", context="talk")

print("环境初始化完成。")
print(f"seaborn 可用：{HAS_SEABORN}")
print("这一步说明什么：后面所有图和表都会复用这里的绘图风格、中文显示和通用库。")


## 0. 配置区

这一节只改最上面的参数，不要急着改后面的分析逻辑。

推荐使用方式：

- 如果你只想分析当前仓库默认结果，先保持外部路径为 `None`
- 如果你在服务器上还有额外文件，就把对应路径填进去
- 如果你的目标列不是 `delta_E`，请优先修改 `TARGET_COL`


In [ ]:
# 这一个代码块集中管理 notebook 的所有可配置项，后面尽量不写死路径和列名。
def find_project_root(start: Path | None = None) -> Path:
    # 允许 notebook 从仓库根目录或 docs 目录启动，并自动向上寻找项目根目录。
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "minimal_adl_ethene_butadiene").exists():
            return candidate
    return start


PROJECT_ROOT = find_project_root()
ADL_ROOT = PROJECT_ROOT / "minimal_adl_ethene_butadiene"

MODEL_VIEW_OPTIONS = {
    "main": {
        "history": ADL_ROOT / "models" / "train_main_history.json",
        "prediction": ADL_ROOT / "models" / "train_main_predictions.csv",
    },
    "aux": {
        "history": ADL_ROOT / "models" / "train_aux_history.json",
        "prediction": ADL_ROOT / "models" / "train_aux_predictions.csv",
    },
}

CONFIG = {
    "PROJECT_ROOT": PROJECT_ROOT,
    "ADL_ROOT": ADL_ROOT,
    "MODEL_VIEW": "main",
    "FEATURE_TABLE_PATH": None,
    "HISTORY_PATH": None,
    "PREDICTION_PATH": None,
    "MODEL_PATH": None,
    "ID_COL": "sample_id",
    "TARGET_COL": None,
    "TRUE_COL": "y_true",
    "PRED_COL": "y_pred",
    "TASK_TYPE": "regression",
    "TOP_N_ERROR": 10,
    "MAX_PLOT_COLUMNS": 12,
    "MAX_HEATMAP_COLUMNS": 20,
    "HIGH_MISSING_THRESHOLD": 0.30,
    "HIGH_CORR_THRESHOLD": 0.90,
    "HIGH_SKEW_THRESHOLD": 1.0,
    "OUTLIER_RATIO_THRESHOLD": 0.05,
}

model_view = CONFIG["MODEL_VIEW"]
if model_view in MODEL_VIEW_OPTIONS:
    CONFIG["HISTORY_PATH"] = CONFIG["HISTORY_PATH"] or MODEL_VIEW_OPTIONS[model_view]["history"]
    CONFIG["PREDICTION_PATH"] = CONFIG["PREDICTION_PATH"] or MODEL_VIEW_OPTIONS[model_view]["prediction"]

STANDARD_PATHS = {
    "base_yaml": ADL_ROOT / "configs" / "base.yaml",
    "delta_dataset_npz": ADL_ROOT / "data" / "processed" / "delta_dataset.npz",
    "delta_dataset_metadata": ADL_ROOT / "data" / "processed" / "delta_dataset_metadata.json",
    "train_main_status": ADL_ROOT / "models" / "train_main_status.json",
    "train_aux_status": ADL_ROOT / "models" / "train_aux_status.json",
    "training_summary": ADL_ROOT / "models" / "training_summary.json",
    "training_state": ADL_ROOT / "models" / "training_state.json",
    "training_split": ADL_ROOT / "models" / "training_split.json",
    "train_main_predictions": ADL_ROOT / "models" / "train_main_predictions.csv",
    "train_aux_predictions": ADL_ROOT / "models" / "train_aux_predictions.csv",
    "train_main_history": ADL_ROOT / "models" / "train_main_history.json",
    "train_aux_history": ADL_ROOT / "models" / "train_aux_history.json",
    "training_diagnostics": ADL_ROOT / "models" / "training_diagnostics.json",
    "uncertainty_latest": ADL_ROOT / "results" / "uncertainty_latest.json",
    "round_001_selection_summary": ADL_ROOT / "results" / "round_001_selection_summary.json",
    "pipeline_run_summary": ADL_ROOT / "results" / "pipeline_run_summary.json",
    "check_environment_report": ADL_ROOT / "results" / "check_environment_latest.json",
}


def load_external_model(model_path: str | Path | None = None):
    # 这里是给服务器侧扩展分析预留的模型加载钩子。
    model_path = model_path or CONFIG["MODEL_PATH"]
    if not model_path:
        return None

    model_path = Path(model_path)
    if not model_path.exists():
        print(f"未找到外部模型文件：{model_path}")
        return None

    suffix = model_path.suffix.lower()
    if suffix in {".pkl", ".pickle"}:
        with model_path.open("rb") as handle:
            return pickle.load(handle)

    if suffix == ".joblib":
        try:
            import joblib
        except Exception as exc:  # noqa: BLE001
            print(f"检测到 joblib 模型文件，但当前环境无法导入 joblib：{exc}")
            return None
        return joblib.load(model_path)

    print("当前默认加载器只支持 .pkl/.pickle/.joblib；如果你的模型格式不同，请手动修改 load_external_model()。")
    return None


print("项目根目录：", CONFIG["PROJECT_ROOT"])
print("ADL 主目录：", CONFIG["ADL_ROOT"])
print("当前默认分析视角：", CONFIG["MODEL_VIEW"])
print("默认 history 文件：", CONFIG["HISTORY_PATH"])
print("默认 prediction 文件：", CONFIG["PREDICTION_PATH"])
print("提示：标准路径下通常不需要先手改 CONFIG；只有你把结果放在别的位置时才需要改。")
print("这一步说明什么：后面所有模块都会读这里的配置，因此迁移到服务器时通常只改这一格。")


## 1. 环境与文件检查

这一节的目标不是“立刻报错”，而是先把能读到什么、缺了什么说清楚。

我们会把文件分成三类：

- `仓库标准文件`：这套最小 ADL 流程默认会写出的文件
- `服务器扩展文件`：为了做更完整机器学习分析，你可以额外提供的文件
- `可选解释文件`：例如树模型文件、SHAP 相关输入，它们缺失时不阻塞主流程


In [ ]:
# 这一个代码块负责路径解析、文件读取和基础状态检查。
def resolve_optional_path(path_like: Any) -> Path | None:
    # 把 None、空字符串、相对路径统一处理成 Path 或 None。
    if path_like in (None, "", "None"):
        return None
    path = Path(path_like)
    if path.is_absolute():
        return path
    return (CONFIG["PROJECT_ROOT"] / path).resolve()


def safe_read_json(path: Path | None, default: Any = None) -> Any:
    # 安全读取 JSON；文件不存在时直接返回默认值，不让 notebook 中断。
    if path is None or not path.exists():
        return default
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except Exception as exc:  # noqa: BLE001
        print(f"读取 JSON 失败：{path} -> {exc}")
        return default


def safe_read_yaml(path: Path | None, default: Any = None) -> Any:
    # 安全读取 YAML；如果环境里文件还没准备好，就返回默认值。
    if path is None or not path.exists():
        return default
    try:
        return yaml.safe_load(path.read_text(encoding="utf-8"))
    except Exception as exc:  # noqa: BLE001
        print(f"读取 YAML 失败：{path} -> {exc}")
        return default


def safe_read_npz(path: Path | None) -> dict[str, np.ndarray]:
    # 把 npz 统一转成字典，便于后续分析。
    if path is None or not path.exists():
        return {}
    try:
        with np.load(path, allow_pickle=True) as payload:
            return {key: payload[key] for key in payload.files}
    except Exception as exc:  # noqa: BLE001
        print(f"读取 NPZ 失败：{path} -> {exc}")
        return {}


def safe_read_table(path: Path | None) -> pd.DataFrame:
    # 兼容 csv / parquet / json / pkl / pickle / npz 等常见表格格式。
    if path is None or not path.exists():
        return pd.DataFrame()

    suffix = path.suffix.lower()
    try:
        if suffix == ".csv":
            return pd.read_csv(path)
        if suffix == ".parquet":
            return pd.read_parquet(path)
        if suffix in {".json", ".jsonl"}:
            try:
                return pd.read_json(path)
            except ValueError:
                return pd.read_json(path, lines=True)
        if suffix in {".pkl", ".pickle"}:
            return pd.read_pickle(path)
        if suffix == ".npz":
            npz_payload = safe_read_npz(path)
            if not npz_payload:
                return pd.DataFrame()
            normalized = {}
            for key, value in npz_payload.items():
                arr = np.asarray(value)
                if arr.ndim == 1:
                    normalized[key] = arr
            return pd.DataFrame(normalized)
    except Exception as exc:  # noqa: BLE001
        print(f"读取表格失败：{path} -> {exc}")
        return pd.DataFrame()

    print(f"暂不支持自动读取这种文件格式：{path.suffix}，请手动扩展 safe_read_table()。")
    return pd.DataFrame()


def safe_read_history(path: Path | None) -> dict[str, list[float]]:
    # history 可以来自 json / csv / pkl；最终都整理成字典形式。
    if path is None or not path.exists():
        return {}

    suffix = path.suffix.lower()
    try:
        if suffix in {".json", ".jsonl"}:
            payload = safe_read_json(path, default={})
            if isinstance(payload, dict) and isinstance(payload.get("history"), dict):
                payload = payload["history"]
            if isinstance(payload, dict):
                return {
                    str(key): [float(item) for item in np.ravel(value).tolist()]
                    for key, value in payload.items()
                    if isinstance(value, (list, tuple, np.ndarray))
                }

        if suffix == ".csv":
            frame = pd.read_csv(path)
            return {
                str(column): frame[column].dropna().tolist()
                for column in frame.columns
            }

        if suffix in {".pkl", ".pickle"}:
            payload = pd.read_pickle(path)
            if hasattr(payload, "history") and isinstance(payload.history, dict):
                payload = payload.history
            if isinstance(payload, dict):
                return {
                    str(key): [float(item) for item in np.ravel(value).tolist()]
                    for key, value in payload.items()
                }
    except Exception as exc:  # noqa: BLE001
        print(f"读取 history 失败：{path} -> {exc}")
        return {}

    print("history 文件已找到，但当前未能自动转换成逐 epoch 字典。")
    return {}


def normalize_sample_id_array(values: Any) -> list[str]:
    # 有些 npz 里的 sample_id 会以 bytes 或 object 存储，这里统一转成字符串。
    normalized = []
    for item in np.asarray(values).tolist():
        if isinstance(item, bytes):
            normalized.append(item.decode("utf-8"))
        else:
            normalized.append(str(item))
    return normalized


def module_available(module_name: str) -> bool:
    # 检查可选依赖是否安装，例如 shap。
    return importlib.util.find_spec(module_name) is not None


OPTIONAL_PATHS = {
    "feature_table": resolve_optional_path(CONFIG["FEATURE_TABLE_PATH"]),
    "history": resolve_optional_path(CONFIG["HISTORY_PATH"]),
    "prediction": resolve_optional_path(CONFIG["PREDICTION_PATH"]),
    "model": resolve_optional_path(CONFIG["MODEL_PATH"]),
}

base_config = safe_read_yaml(STANDARD_PATHS["base_yaml"], default={}) or {}
delta_dataset = safe_read_npz(STANDARD_PATHS["delta_dataset_npz"])
delta_metadata = safe_read_json(STANDARD_PATHS["delta_dataset_metadata"], default={}) or {}
train_main_status = safe_read_json(STANDARD_PATHS["train_main_status"], default={}) or {}
train_aux_status = safe_read_json(STANDARD_PATHS["train_aux_status"], default={}) or {}
training_summary = safe_read_json(STANDARD_PATHS["training_summary"], default={}) or {}
training_state = safe_read_json(STANDARD_PATHS["training_state"], default={}) or {}
training_split = safe_read_json(STANDARD_PATHS["training_split"], default={}) or {}
training_diagnostics = safe_read_json(STANDARD_PATHS["training_diagnostics"], default={}) or {}
pipeline_summary = safe_read_json(STANDARD_PATHS["pipeline_run_summary"], default={}) or {}
uncertainty_payload = safe_read_json(STANDARD_PATHS["uncertainty_latest"], default={}) or {}
selection_summary = safe_read_json(STANDARD_PATHS["round_001_selection_summary"], default={}) or {}

external_feature_df = safe_read_table(OPTIONAL_PATHS["feature_table"])
external_prediction_df = safe_read_table(OPTIONAL_PATHS["prediction"])
history_raw_payload = safe_read_json(OPTIONAL_PATHS["history"], default={}) or {}
history_payload = safe_read_history(OPTIONAL_PATHS["history"])

file_rows = []
for name, path in STANDARD_PATHS.items():
    file_rows.append(
        {
            "类别": "仓库标准文件",
            "文件键": name,
            "路径": str(path),
            "是否存在": path.exists(),
            "是否必需": name in {"base_yaml", "delta_dataset_metadata", "training_summary", "training_diagnostics", "uncertainty_latest"},
        }
    )

for name, path in OPTIONAL_PATHS.items():
    file_rows.append(
        {
            "类别": "可配置分析输入",
            "文件键": name,
            "路径": "" if path is None else str(path),
            "是否存在": False if path is None else path.exists(),
            "是否必需": False,
        }
    )

file_status_df = pd.DataFrame(file_rows)
display(file_status_df)

print("可选依赖检查：")
print(f"- seaborn 是否可用：{HAS_SEABORN}")
print(f"- shap 是否可用：{module_available('shap')}")
print(f"- scikit-learn 是否可用：{module_available('sklearn')}")

if isinstance(training_diagnostics, dict) and training_diagnostics:
    print("检测到 training_diagnostics.json，说明训练阶段已经按新标准导出了 notebook 所需的分析入口。")
if isinstance(pipeline_summary, dict) and pipeline_summary:
    print(f"检测到 pipeline_run_summary.json，最近一次主控流程 success={pipeline_summary.get('success')}")

if not file_status_df["是否存在"].all():
    missing_df = file_status_df.loc[~file_status_df["是否存在"], ["类别", "文件键", "路径", "是否必需"]]
    display(missing_df)
    print("提示：缺失文件不会让整本 notebook 直接崩掉；后续对应模块会给出跳过说明。")

print("这一步说明什么：先确认当前到底能分析哪些内容，避免把“缺文件”误判成“代码出错”。")


## 2. 数据长什么样？

这一节重点回答：

- 数据规模有多大
- 每一列分别是什么类型
- 有没有缺失值、异常值、偏态和高度相关
- 这些问题会不会影响后续建模

如果服务器上已经准备好了外部特征表，就优先用外部特征表做 EDA；否则就从 ADL 的 `delta_dataset.npz` 和 metadata 中构造一个可分析 DataFrame。


In [ ]:
# 这一个代码块负责构造分析 DataFrame，并完成完整的 EDA。
def flatten_metadata_field(payload: Any, prefix: str = "meta") -> dict[str, Any]:
    # 把 metadata 中的嵌套字典尽量展开，方便后续做表格分析。
    if not isinstance(payload, dict):
        return {}
    flattened = {}
    for key, value in payload.items():
        flat_key = f"{prefix}_{key}"
        if isinstance(value, dict):
            for inner_key, inner_value in value.items():
                flattened[f"{flat_key}_{inner_key}"] = inner_value
        else:
            flattened[flat_key] = value
    return flattened


def build_adl_analysis_df(dataset_payload: dict[str, np.ndarray], metadata_payload: dict[str, Any]) -> pd.DataFrame:
    # 根据仓库标准输出构造一个更适合数据分析的样本级表格。
    if not dataset_payload:
        return pd.DataFrame()

    sample_ids = normalize_sample_id_array(dataset_payload.get("sample_ids", []))
    total_samples = len(sample_ids)
    if total_samples == 0:
        return pd.DataFrame()

    records = []
    sample_metadata_list = metadata_payload.get("samples", []) if isinstance(metadata_payload, dict) else []
    metadata_by_id = {
        str(item.get("sample_id")): item
        for item in sample_metadata_list
        if isinstance(item, dict) and item.get("sample_id") is not None
    }

    atomic_numbers = np.asarray(dataset_payload.get("atomic_numbers", []), dtype=object)
    coordinates = np.asarray(dataset_payload.get("coordinates", []), dtype=object)
    e_baseline = np.asarray(dataset_payload.get("E_baseline", np.full(total_samples, np.nan)), dtype=float)
    e_target = np.asarray(dataset_payload.get("E_target", np.full(total_samples, np.nan)), dtype=float)
    delta_e = np.asarray(dataset_payload.get("delta_E", np.full(total_samples, np.nan)), dtype=float)
    f_baseline = np.asarray(dataset_payload.get("F_baseline", np.zeros((total_samples, 1, 3))), dtype=float)
    f_target = np.asarray(dataset_payload.get("F_target", np.zeros((total_samples, 1, 3))), dtype=float)
    delta_f = np.asarray(dataset_payload.get("delta_F", np.zeros((total_samples, 1, 3))), dtype=float)

    for idx, sample_id in enumerate(sample_ids):
        sample_meta = metadata_by_id.get(sample_id, {})
        atomic_number_array = np.asarray(atomic_numbers[idx]) if idx < len(atomic_numbers) else np.asarray([])
        coordinate_array = np.asarray(coordinates[idx]) if idx < len(coordinates) else np.asarray([])
        record = {
            CONFIG["ID_COL"]: sample_id,
            "E_baseline": e_baseline[idx] if idx < len(e_baseline) else np.nan,
            "E_target": e_target[idx] if idx < len(e_target) else np.nan,
            "delta_E": delta_e[idx] if idx < len(delta_e) else np.nan,
            "|F_baseline|": float(np.linalg.norm(np.ravel(f_baseline[idx]))) if idx < len(f_baseline) else np.nan,
            "|F_target|": float(np.linalg.norm(np.ravel(f_target[idx]))) if idx < len(f_target) else np.nan,
            "|delta_F|": float(np.linalg.norm(np.ravel(delta_f[idx]))) if idx < len(delta_f) else np.nan,
            "num_atoms": int(len(atomic_number_array)),
            "charge": sample_meta.get("charge"),
            "multiplicity": sample_meta.get("multiplicity"),
            "source": sample_meta.get("source", ""),
            "geometry_file": sample_meta.get("geometry_file", ""),
            "mean_atomic_number": float(np.mean(atomic_number_array)) if atomic_number_array.size else np.nan,
            "coordinate_std": float(np.std(coordinate_array)) if coordinate_array.size else np.nan,
        }
        record.update(flatten_metadata_field(sample_meta.get("manifest_metadata", {}), prefix="manifest"))
        records.append(record)

    return pd.DataFrame(records)


def merge_external_and_adl(feature_df: pd.DataFrame, adl_df: pd.DataFrame) -> tuple[pd.DataFrame, str]:
    # 如果同时存在外部特征表和 ADL 标准表，则优先用外部表，并按 sample_id 尽量补齐 ADL 字段。
    if feature_df.empty and adl_df.empty:
        return pd.DataFrame(), "未检测到可分析的数据表。"
    if feature_df.empty:
        return adl_df.copy(), "当前使用 ADL 标准结果构造的数据表。"
    if adl_df.empty:
        return feature_df.copy(), "当前使用服务器提供的外部特征表。"

    left = feature_df.copy()
    right = adl_df.copy()
    if CONFIG["ID_COL"] in left.columns and CONFIG["ID_COL"] in right.columns:
        merged = left.merge(right, on=CONFIG["ID_COL"], how="left", suffixes=("", "_adl"))
        return merged, "当前以服务器外部特征表为主，并补充了 ADL 标准字段。"
    return left, "检测到外部特征表，但没有可用于合并的 sample_id，因此仅分析外部特征表。"


def detect_column_groups(frame: pd.DataFrame) -> tuple[list[str], list[str]]:
    # 自动区分数值列和类别列，便于分别处理。
    if frame.empty:
        return [], []
    numeric_cols = frame.select_dtypes(include=[np.number, "bool"]).columns.tolist()
    categorical_cols = [column for column in frame.columns if column not in numeric_cols]
    return numeric_cols, categorical_cols


def infer_target_column(frame: pd.DataFrame) -> str | None:
    # 优先使用用户显式设置的 TARGET_COL；没有时尝试按 ADL 默认目标 delta_E 推断。
    if CONFIG["TARGET_COL"] and CONFIG["TARGET_COL"] in frame.columns:
        return CONFIG["TARGET_COL"]
    if "delta_E" in frame.columns:
        return "delta_E"
    if CONFIG["TRUE_COL"] in frame.columns:
        return CONFIG["TRUE_COL"]
    return None


def summarize_outliers(frame: pd.DataFrame, numeric_cols: list[str]) -> pd.DataFrame:
    # 用 IQR 粗略统计每个数值特征的异常值比例。
    rows = []
    for column in numeric_cols:
        series = frame[column].dropna()
        if series.empty:
            continue
        q1 = series.quantile(0.25)
        q3 = series.quantile(0.75)
        iqr = q3 - q1
        if iqr == 0:
            outlier_ratio = 0.0
        else:
            lower = q1 - 1.5 * iqr
            upper = q3 + 1.5 * iqr
            outlier_ratio = ((series < lower) | (series > upper)).mean()
        rows.append({"特征": column, "异常值比例": outlier_ratio})
    return pd.DataFrame(rows).sort_values("异常值比例", ascending=False)


def plot_missingness(frame: pd.DataFrame) -> None:
    # 画出各列缺失率以及缺失模式热图。
    if frame.empty:
        print("当前没有可分析的数据表，跳过缺失值可视化。")
        return

    missing_rate = frame.isna().mean().sort_values(ascending=False)
    plt.figure(figsize=(10, 5))
    missing_rate.plot(kind="bar", color="#D55E00")
    plt.title("各字段缺失率")
    plt.xlabel("字段名")
    plt.ylabel("缺失比例")
    plt.xticks(rotation=60, ha="right")
    plt.tight_layout()
    plt.show()
    print("这张图说明什么：可以快速看出哪些字段缺失严重，决定后续是补值、删列还是追查数据来源。")

    subset_cols = missing_rate.head(min(30, len(missing_rate))).index.tolist()
    plt.figure(figsize=(12, max(4, len(frame) * 0.03)))
    sns.heatmap(frame[subset_cols].isna(), cbar=True, yticklabels=False, cmap="YlOrRd")
    plt.title("缺失模式热图（最多展示缺失率最高的 30 列）")
    plt.xlabel("字段名")
    plt.ylabel("样本")
    plt.tight_layout()
    plt.show()
    print("这张图说明什么：它能帮助判断缺失是随机出现，还是集中在某些样本或某些字段组合上。")


def plot_numeric_histograms(frame: pd.DataFrame, numeric_cols: list[str]) -> None:
    # 直方图用于观察数值特征的分布形状、偏态和长尾现象。
    if not numeric_cols:
        print("没有数值特征，跳过直方图。")
        return

    selected_cols = numeric_cols[: CONFIG["MAX_PLOT_COLUMNS"]]
    n_cols = 3
    n_rows = math.ceil(len(selected_cols) / n_cols)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 4 * n_rows))
    axes = np.array(axes).reshape(-1)
    for ax, column in zip(axes, selected_cols):
        sns.histplot(frame[column].dropna(), kde=True, ax=ax, color="#0072B2")
        ax.set_title(f"{column} 分布")
        ax.set_xlabel(column)
        ax.set_ylabel("频数")
    for ax in axes[len(selected_cols):]:
        ax.axis("off")
    fig.suptitle("数值特征直方图", y=1.02)
    plt.tight_layout()
    plt.show()
    print("这张图说明什么：可以观察特征是否近似对称、是否存在长尾，以及是否需要做对数变换或标准化。")


def plot_boxplots(frame: pd.DataFrame, numeric_cols: list[str]) -> None:
    # 箱线图适合检查离群点和分布范围。
    if not numeric_cols:
        print("没有数值特征，跳过箱线图。")
        return

    selected_cols = numeric_cols[: CONFIG["MAX_PLOT_COLUMNS"]]
    plt.figure(figsize=(max(10, len(selected_cols) * 1.1), 6))
    sns.boxplot(data=frame[selected_cols], orient="h")
    plt.title("数值特征箱线图")
    plt.xlabel("取值")
    plt.ylabel("特征")
    plt.tight_layout()
    plt.show()
    print("这张图说明什么：如果箱线图中离群点很多，说明后续模型可能需要更稳健的缩放、截断或异常值处理。")


def plot_correlation_heatmap(frame: pd.DataFrame, numeric_cols: list[str]) -> None:
    # 热力图用于检查特征之间是否高度相关，辅助判断共线性问题。
    if len(numeric_cols) < 2:
        print("数值特征少于 2 列，跳过相关系数热力图。")
        return

    selected_cols = numeric_cols[: CONFIG["MAX_HEATMAP_COLUMNS"]]
    corr = frame[selected_cols].corr(numeric_only=True)
    plt.figure(figsize=(max(8, len(selected_cols) * 0.7), max(6, len(selected_cols) * 0.6)))
    sns.heatmap(corr, cmap="coolwarm", center=0, annot=False, square=False)
    plt.title("数值特征相关系数热力图")
    plt.xlabel("特征")
    plt.ylabel("特征")
    plt.tight_layout()
    plt.show()
    print("这张图说明什么：如果某些特征高度相关，建模时可能出现信息重复、解释不稳定或参数冗余。")


def plot_categorical_distributions(frame: pd.DataFrame, categorical_cols: list[str]) -> None:
    # 类别分布图适合快速判断类别是否稀疏、是否失衡。
    if not categorical_cols:
        print("没有类别变量，跳过类别分布图。")
        return

    selected_cols = categorical_cols[: min(6, len(categorical_cols))]
    n_rows = math.ceil(len(selected_cols) / 2)
    fig, axes = plt.subplots(n_rows, 2, figsize=(12, 4 * n_rows))
    axes = np.array(axes).reshape(-1)
    for ax, column in zip(axes, selected_cols):
        counts = frame[column].fillna("缺失").astype(str).value_counts().head(12)
        sns.barplot(x=counts.values, y=counts.index, ax=ax, color="#009E73")
        ax.set_title(f"{column} 类别分布")
        ax.set_xlabel("样本数")
        ax.set_ylabel(column)
    for ax in axes[len(selected_cols):]:
        ax.axis("off")
    fig.suptitle("类别变量分布图", y=1.02)
    plt.tight_layout()
    plt.show()
    print("这张图说明什么：如果某些类别样本极少，后续建模与解释都要警惕类别稀疏带来的不稳定性。")


def plot_target_distribution(frame: pd.DataFrame, target_col: str | None) -> None:
    # 这里默认按回归任务画目标变量分布图。
    if target_col is None or target_col not in frame.columns:
        print("未找到目标列，跳过目标变量分布图。")
        return

    plt.figure(figsize=(8, 5))
    sns.histplot(frame[target_col].dropna(), kde=True, color="#CC79A7")
    plt.title(f"目标变量 {target_col} 分布")
    plt.xlabel(target_col)
    plt.ylabel("频数")
    plt.tight_layout()
    plt.show()
    print("这张图说明什么：它帮助判断目标值是否集中、偏态或长尾，从而影响损失函数和采样策略。")


def summarize_eda_issues(frame: pd.DataFrame, numeric_cols: list[str], target_col: str | None) -> dict[str, Any]:
    # 自动汇总 EDA 中最值得注意的问题，并给出建模前预处理建议。
    summary = {
        "high_missing_cols": [],
        "high_skew_cols": [],
        "high_corr_pairs": [],
        "high_outlier_cols": [],
        "preprocess_suggestions": [],
    }
    if frame.empty:
        return summary

    missing_rate = frame.isna().mean()
    summary["high_missing_cols"] = missing_rate[missing_rate >= CONFIG["HIGH_MISSING_THRESHOLD"]].index.tolist()

    if numeric_cols:
        skew_series = frame[numeric_cols].skew(numeric_only=True).sort_values(key=np.abs, ascending=False)
        summary["high_skew_cols"] = skew_series[skew_series.abs() >= CONFIG["HIGH_SKEW_THRESHOLD"]].index.tolist()

        corr = frame[numeric_cols].corr(numeric_only=True).abs()
        for i, left_col in enumerate(corr.columns):
            for right_col in corr.columns[i + 1:]:
                value = corr.loc[left_col, right_col]
                if pd.notna(value) and value >= CONFIG["HIGH_CORR_THRESHOLD"]:
                    summary["high_corr_pairs"].append((left_col, right_col, float(value)))

        outlier_df = summarize_outliers(frame, numeric_cols)
        if not outlier_df.empty:
            summary["high_outlier_cols"] = outlier_df.loc[
                outlier_df["异常值比例"] >= CONFIG["OUTLIER_RATIO_THRESHOLD"], "特征"
            ].tolist()

    if summary["high_missing_cols"]:
        summary["preprocess_suggestions"].append("对缺失严重的字段先判断是否保留；必要时删列或采用分组补值。")
    if summary["high_skew_cols"]:
        summary["preprocess_suggestions"].append("对偏态明显的连续变量考虑对数变换、Box-Cox 或更稳健的缩放。")
    if summary["high_corr_pairs"]:
        summary["preprocess_suggestions"].append("对高度相关的特征考虑删减、合并或在解释时成组看待。")
    if summary["high_outlier_cols"]:
        summary["preprocess_suggestions"].append("对异常值比例高的字段考虑截尾、稳健模型或单独检查上游标注。")
    if target_col is not None and target_col in frame.columns and frame[target_col].dropna().skew() > CONFIG["HIGH_SKEW_THRESHOLD"]:
        summary["preprocess_suggestions"].append("目标变量偏态较强，可考虑对目标做变换，或在报告中解释高值样本更难拟合。")
    if not summary["preprocess_suggestions"]:
        summary["preprocess_suggestions"].append("当前未发现特别突出的数据问题，可以先用标准化 + 常规回归流程作为基线。")
    return summary


adl_analysis_df = build_adl_analysis_df(delta_dataset, delta_metadata)
analysis_df, data_source_note = merge_external_and_adl(external_feature_df, adl_analysis_df)
numeric_columns, categorical_columns = detect_column_groups(analysis_df)
effective_target_col = infer_target_column(analysis_df)

print(data_source_note)
if analysis_df.empty:
    print("当前没有可分析的数据集。请在服务器环境中准备 FEATURE_TABLE_PATH，或确保 ADL 标准结果文件存在。")
    eda_summary = {
        "high_missing_cols": [],
        "high_skew_cols": [],
        "high_corr_pairs": [],
        "high_outlier_cols": [],
        "preprocess_suggestions": ["请先准备数据文件，再运行 EDA。"],
    }
else:
    overview_df = pd.DataFrame(
        {
            "指标": ["样本数", "字段数", "数值列数", "类别列数", "目标列"],
            "取值": [
                analysis_df.shape[0],
                analysis_df.shape[1],
                len(numeric_columns),
                len(categorical_columns),
                effective_target_col or "未识别",
            ],
        }
    )
    display(overview_df)

    column_profile_df = pd.DataFrame(
        {
            "字段名": analysis_df.columns,
            "数据类型": analysis_df.dtypes.astype(str).values,
            "缺失比例": analysis_df.isna().mean().values,
            "非空样本数": analysis_df.notna().sum().values,
        }
    ).sort_values(["缺失比例", "字段名"], ascending=[False, True])
    display(column_profile_df)

    print("这张表说明什么：它先回答“数据有多大、字段是什么类型、哪些列最容易出问题”。")

    plot_missingness(analysis_df)

    if numeric_columns:
        numeric_stats_df = analysis_df[numeric_columns].describe().T
        numeric_stats_df["skewness"] = analysis_df[numeric_columns].skew(numeric_only=True)
        display(numeric_stats_df)
        print("这张表说明什么：它概括了数值特征的中心位置、离散程度和偏态，是后续缩放与变换的依据。")

    plot_numeric_histograms(analysis_df, numeric_columns)
    plot_boxplots(analysis_df, numeric_columns)

    if numeric_columns:
        skew_df = (
            analysis_df[numeric_columns]
            .skew(numeric_only=True)
            .rename("skewness")
            .sort_values(key=np.abs, ascending=False)
            .reset_index()
            .rename(columns={"index": "特征"})
        )
        display(skew_df)
        print("这张表说明什么：偏态绝对值越大，说明分布越不对称，模型训练时越可能受极端值影响。")

    plot_correlation_heatmap(analysis_df, numeric_columns)
    plot_categorical_distributions(analysis_df, categorical_columns)
    plot_target_distribution(analysis_df, effective_target_col)

    outlier_summary_df = summarize_outliers(analysis_df, numeric_columns)
    if not outlier_summary_df.empty:
        display(outlier_summary_df)
        print("这张表说明什么：它统计了每个数值特征的粗略异常值比例，便于判断哪些字段最值得重点清洗。")

    eda_summary = summarize_eda_issues(analysis_df, numeric_columns, effective_target_col)
    eda_conclusion_lines = []
    if eda_summary["high_missing_cols"]:
        eda_conclusion_lines.append(f"- 缺失严重的字段：{', '.join(eda_summary['high_missing_cols'])}")
    if eda_summary["high_outlier_cols"]:
        eda_conclusion_lines.append(f"- 异常值比例较高的字段：{', '.join(eda_summary['high_outlier_cols'])}")
    if eda_summary["high_skew_cols"]:
        eda_conclusion_lines.append(f"- 偏态明显的字段：{', '.join(eda_summary['high_skew_cols'][:10])}")
    if eda_summary["high_corr_pairs"]:
        preview_pairs = [f"{left} / {right} ({value:.2f})" for left, right, value in eda_summary["high_corr_pairs"][:8]]
        eda_conclusion_lines.append(f"- 高度相关的特征对：{'; '.join(preview_pairs)}")

    if not eda_conclusion_lines:
        eda_conclusion_lines.append("- 当前没有发现特别突出的缺失、偏态、异常值或共线性问题。")

    display(Markdown("### EDA 自动发现的问题\n" + "\n".join(eda_conclusion_lines)))
    display(Markdown("### 建模前建议做的预处理\n" + "\n".join(f"- {item}" for item in eda_summary["preprocess_suggestions"])))
    print("这一步说明什么：EDA 不只是画图，还要把会影响建模的风险点提前暴露出来。")


## 3. 模型训练得顺不顺？

这一节分两层看：

- `最终结果层`：先看主模型和辅助模型的 train/validation 指标
- `训练过程层`：如果服务器上保留了逐 epoch 的 history，再判断是否收敛、是否过拟合或欠拟合

如果 history 不存在，本 notebook 会清楚提示“只能看最终指标，不能看训练曲线”。


In [ ]:
# 这一个代码块负责整理训练摘要、绘制 history 曲线，并自动输出中文训练结论。
def training_summary_to_table(summary_payload: dict[str, Any]) -> pd.DataFrame:
    # 把 training_summary.json 统一整理成长表，便于画柱状图和对比主模型/辅助模型。
    if not isinstance(summary_payload, dict) or not summary_payload:
        return pd.DataFrame()

    rows = []
    for model_name in ["main_model", "aux_model"]:
        model_metrics = summary_payload.get(model_name)
        if not isinstance(model_metrics, dict):
            continue
        for metric_name, metric_value in model_metrics.items():
            rows.append(
                {
                    "模型": "主模型" if model_name == "main_model" else "辅助模型",
                    "原始模型键": model_name,
                    "指标名": metric_name,
                    "数值": metric_value,
                }
            )
    return pd.DataFrame(rows)


def pick_history_series(history: dict[str, list[float]], aliases: list[str]) -> list[float]:
    # 同一个指标常常有多个命名版本，这里按别名顺序自动兼容。
    for alias in aliases:
        if alias in history and history[alias]:
            return [float(item) for item in history[alias]]
    return []


def normalize_history(history: dict[str, list[float]]) -> dict[str, list[float]]:
    # 把 history 字典里的值尽量都转换成 float 列表，方便统一分析。
    normalized = {}
    if not isinstance(history, dict):
        return normalized
    for key, value in history.items():
        try:
            normalized[str(key)] = [float(item) for item in value]
        except Exception:
            continue
    return normalized


def analyze_training_history(history: dict[str, list[float]], stable_window: int = 5) -> dict[str, Any]:
    # 根据训练/验证曲线自动判断收敛、过拟合和欠拟合。
    history = normalize_history(history)
    train_loss = pick_history_series(history, ["train_loss", "loss"])
    val_loss = pick_history_series(history, ["val_loss", "valid_loss", "validation_loss"])

    result = {
        "has_history": bool(train_loss or val_loss),
        "is_converged": None,
        "possible_overfitting": None,
        "possible_underfitting": None,
        "best_epoch": None,
        "best_val_loss": None,
        "conclusion_lines": [],
    }

    if not train_loss and not val_loss:
        result["conclusion_lines"].append("当前没有逐 epoch history，只能根据最终指标判断训练结果，无法观察完整收敛过程。")
        return result

    if val_loss:
        best_idx = int(np.argmin(val_loss))
        result["best_epoch"] = best_idx + 1
        result["best_val_loss"] = float(val_loss[best_idx])
    elif train_loss:
        best_idx = int(np.argmin(train_loss))
        result["best_epoch"] = best_idx + 1
        result["best_val_loss"] = float(train_loss[best_idx])

    tail = val_loss[-stable_window:] if len(val_loss) >= stable_window else val_loss
    if tail:
        tail_range = max(tail) - min(tail)
        baseline = max(abs(np.mean(tail)), 1e-8)
        result["is_converged"] = tail_range / baseline < 0.05

    if train_loss and val_loss:
        min_val = min(val_loss)
        result["possible_overfitting"] = val_loss[-1] > min_val * 1.05 and train_loss[-1] < train_loss[0] * 0.8
        train_improvement = (train_loss[0] - train_loss[-1]) / max(abs(train_loss[0]), 1e-8)
        val_improvement = (val_loss[0] - val_loss[-1]) / max(abs(val_loss[0]), 1e-8)
        result["possible_underfitting"] = train_improvement < 0.10 and val_improvement < 0.10

    if result["is_converged"] is True:
        result["conclusion_lines"].append("验证集 loss 在最后若干个 epoch 已基本趋稳，模型看起来已经基本收敛。")
    elif result["is_converged"] is False:
        result["conclusion_lines"].append("验证集 loss 末段仍有明显波动，说明训练过程还没有完全稳定。")

    if result["possible_overfitting"] is True:
        result["conclusion_lines"].append("训练集 loss 持续下降但验证集 loss 偏离最优点，存在过拟合迹象。")
    if result["possible_underfitting"] is True:
        result["conclusion_lines"].append("训练集和验证集的改善都有限，存在欠拟合或训练不足的可能。")
    if result["best_epoch"] is not None:
        result["conclusion_lines"].append(f"验证集表现最好的 epoch 大约在第 {result['best_epoch']} 轮。")
    if not result["conclusion_lines"]:
        result["conclusion_lines"].append("训练过程整体平稳，但仍建议结合业务目标和最终误差指标一起判断。")
    return result


def plot_history_curves(history: dict[str, list[float]]) -> None:
    # 画 loss 曲线，并按存在的指标自动补充 RMSE/MAE/accuracy 曲线。
    history = normalize_history(history)
    metric_groups = [
        ("loss", ["train_loss", "loss"], ["val_loss", "valid_loss", "validation_loss"]),
        ("rmse", ["train_rmse", "rmse"], ["val_rmse", "valid_rmse", "validation_rmse"]),
        ("mae", ["train_mae", "mae"], ["val_mae", "valid_mae", "validation_mae"]),
        ("accuracy", ["train_acc", "acc", "accuracy"], ["val_acc", "val_accuracy", "valid_accuracy"]),
    ]

    plotted = False
    for metric_label, train_aliases, val_aliases in metric_groups:
        train_series = pick_history_series(history, train_aliases)
        val_series = pick_history_series(history, val_aliases)
        if not train_series and not val_series:
            continue

        plotted = True
        epochs = np.arange(1, max(len(train_series), len(val_series)) + 1)
        plt.figure(figsize=(8, 5))
        if train_series:
            plt.plot(epochs[: len(train_series)], train_series, marker="o", label=f"训练集 {metric_label}")
        if val_series:
            plt.plot(epochs[: len(val_series)], val_series, marker="s", label=f"验证集 {metric_label}")
        plt.title(f"训练过程中的 {metric_label} 曲线")
        plt.xlabel("Epoch")
        plt.ylabel(metric_label)
        plt.legend()
        plt.tight_layout()
        plt.show()
        print(f"这张图说明什么：它用来观察 {metric_label} 是否稳定下降，以及训练集与验证集之间是否逐渐分叉。")

    if not plotted:
        print("history 文件已找到，但没有识别出可绘制的 loss / rmse / mae / accuracy 字段。")


training_metrics_df = training_summary_to_table(training_summary)
history_analysis = analyze_training_history(history_payload)

if training_metrics_df.empty:
    print("未检测到 training_summary.json 或内容为空，无法展示最终训练指标。")
else:
    display(training_metrics_df)
    print("这张表说明什么：它把主模型和辅助模型的训练集/验证集指标放在一起，便于直接横向比较。")

    plot_df = training_metrics_df.copy()
    plt.figure(figsize=(12, 6))
    sns.barplot(data=plot_df, x="指标名", y="数值", hue="模型")
    plt.title("主模型与辅助模型训练指标对比")
    plt.xlabel("指标名")
    plt.ylabel("数值")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()
    print("这张图说明什么：柱状图能帮助快速判断验证集误差是否显著高于训练集，以及主模型和辅助模型谁更稳定。")

history_reason = None
if isinstance(history_raw_payload, dict) and history_raw_payload:
    history_reason = history_raw_payload.get("reason")

if history_payload:
    plot_history_curves(history_payload)
elif OPTIONAL_PATHS["history"] is not None and OPTIONAL_PATHS["history"].exists():
    print(f"history 文件已找到，但当前没有可用的逐 epoch 指标：{history_reason or '未记录具体原因'}")
else:
    print("当前没有找到可用的 history 文件，因此只能分析最终指标，不能判断完整训练曲线。")

display(Markdown("### 自动训练结论\n" + "\n".join(f"- {item}" for item in history_analysis["conclusion_lines"])))

if not training_metrics_df.empty:
    main_validation_energy = training_summary.get("main_model", {}).get("validation_energy_rmse")
    main_validation_gradient = training_summary.get("main_model", {}).get("validation_gradient_rmse")
    main_validation_pcc = training_summary.get("main_model", {}).get("validation_energy_pcc")
    print("如何看这些指标：")
    print(f"- energy RMSE 越小越好，当前主模型验证集约为：{main_validation_energy}")
    print(f"- gradient RMSE 越小越好，当前主模型验证集约为：{main_validation_gradient}")
    print(f"- PCC 越接近 1 越好，当前主模型验证集约为：{main_validation_pcc}")
    print("- 验证集比训练集更重要，因为它更接近模型在新样本上的真实表现。")
    print("- 训练成功不等于主动学习已经收敛，后面还要结合 UQ 和高不确定性比例继续判断。")

training_findings = {
    "history_analysis": history_analysis,
    "training_metrics_df": training_metrics_df,
}


## 4. 模型预测得准不准？

这一节默认按 `回归任务` 分析，重点看：

- `MAE / MSE / RMSE / R²`
- 真实值 vs 预测值
- 残差图和残差分布
- 哪些样本误差最大
- 模型在高值区还是低值区更容易出错

如果没有逐样本预测文件，本节会退化为“只展示 training_summary 中已有的最终指标”。


In [ ]:
# 这一个代码块负责逐样本回归评估与误差分析。
def build_uncertainty_df(payload: dict[str, Any]) -> pd.DataFrame:
    # 把 uncertainty_latest.json 整理成表格，方便与预测结果按 sample_id 合并。
    if not isinstance(payload, dict):
        return pd.DataFrame()
    samples = payload.get("samples", [])
    if not isinstance(samples, list):
        return pd.DataFrame()
    frame = pd.DataFrame(samples)
    if not frame.empty and CONFIG["ID_COL"] not in frame.columns and "sample_id" in frame.columns:
        frame[CONFIG["ID_COL"]] = frame["sample_id"]
    return frame


def prepare_prediction_df(
    prediction_df: pd.DataFrame,
    analysis_frame: pd.DataFrame,
    uq_frame: pd.DataFrame,
) -> tuple[pd.DataFrame, str]:
    # 优先使用服务器侧逐样本预测文件；如果没有，再尝试从分析表中寻找 y_true / y_pred。
    candidate = pd.DataFrame()
    note = ""

    if not prediction_df.empty:
        candidate = prediction_df.copy()
        note = "当前使用服务器提供的逐样本预测文件。"
    elif (
        not analysis_frame.empty
        and CONFIG["TRUE_COL"] in analysis_frame.columns
        and CONFIG["PRED_COL"] in analysis_frame.columns
    ):
        candidate = analysis_frame[[CONFIG["ID_COL"], CONFIG["TRUE_COL"], CONFIG["PRED_COL"]]].copy()
        note = "当前直接使用分析表中已有的真实值与预测值列。"

    if candidate.empty:
        return pd.DataFrame(), "未检测到逐样本预测结果，因此无法做散点图、残差图和误差分桶分析。"

    if CONFIG["ID_COL"] not in candidate.columns:
        candidate[CONFIG["ID_COL"]] = [f"sample_{idx:04d}" for idx in range(len(candidate))]

    if CONFIG["TRUE_COL"] not in candidate.columns or CONFIG["PRED_COL"] not in candidate.columns:
        return pd.DataFrame(), "预测结果表缺少真实值或预测值列，请检查 TRUE_COL / PRED_COL 设置。"

    candidate = candidate.copy()
    candidate[CONFIG["TRUE_COL"]] = pd.to_numeric(candidate[CONFIG["TRUE_COL"]], errors="coerce")
    candidate[CONFIG["PRED_COL"]] = pd.to_numeric(candidate[CONFIG["PRED_COL"]], errors="coerce")
    candidate["residual"] = candidate[CONFIG["PRED_COL"]] - candidate[CONFIG["TRUE_COL"]]
    candidate["abs_error"] = candidate["residual"].abs()

    if not uq_frame.empty and CONFIG["ID_COL"] in uq_frame.columns:
        candidate = candidate.merge(
            uq_frame[[col for col in uq_frame.columns if col in {CONFIG["ID_COL"], "uncertainty", "exceeds_threshold"}]],
            on=CONFIG["ID_COL"],
            how="left",
        )

    return candidate.dropna(subset=[CONFIG["TRUE_COL"], CONFIG["PRED_COL"]]), note


def evaluate_regression(frame: pd.DataFrame) -> dict[str, float]:
    # 汇总常用回归指标。
    if frame.empty:
        return {}
    y_true = frame[CONFIG["TRUE_COL"]].to_numpy()
    y_pred = frame[CONFIG["PRED_COL"]].to_numpy()
    mse = mean_squared_error(y_true, y_pred)
    return {
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "MSE": float(mse),
        "RMSE": float(np.sqrt(mse)),
        "R2": float(r2_score(y_true, y_pred)),
    }


def plot_true_vs_pred(frame: pd.DataFrame) -> None:
    # 理想情况下，散点应该尽量贴近 y=x 参考线。
    if frame.empty:
        return
    y_true = frame[CONFIG["TRUE_COL"]]
    y_pred = frame[CONFIG["PRED_COL"]]
    line_min = min(y_true.min(), y_pred.min())
    line_max = max(y_true.max(), y_pred.max())
    plt.figure(figsize=(7, 7))
    sns.scatterplot(data=frame, x=CONFIG["TRUE_COL"], y=CONFIG["PRED_COL"], s=60, alpha=0.75)
    plt.plot([line_min, line_max], [line_min, line_max], linestyle="--", color="black", label="y = x 参考线")
    plt.title("真实值 vs 预测值")
    plt.xlabel("真实值")
    plt.ylabel("预测值")
    plt.legend()
    plt.tight_layout()
    plt.show()
    print("这张图说明什么：散点越贴近参考线，说明模型整体预测越准确；系统性偏离则提示偏差。")


def plot_residuals(frame: pd.DataFrame) -> None:
    # 残差图常用来检查误差是否随预测值系统性变化。
    if frame.empty:
        return
    plt.figure(figsize=(8, 5))
    sns.scatterplot(data=frame, x=CONFIG["PRED_COL"], y="residual", s=60, alpha=0.75)
    plt.axhline(0, linestyle="--", color="black")
    plt.title("残差图（Residual vs Predicted）")
    plt.xlabel("预测值")
    plt.ylabel("残差（预测值 - 真实值）")
    plt.tight_layout()
    plt.show()
    print("这张图说明什么：如果残差在 0 附近随机散开，说明误差模式较健康；若出现趋势，则可能有系统性偏差。")


def plot_residual_distribution(frame: pd.DataFrame) -> None:
    # 残差分布图帮助判断误差是否偏斜、是否存在长尾。
    if frame.empty:
        return
    plt.figure(figsize=(8, 5))
    sns.histplot(frame["residual"].dropna(), kde=True, color="#E69F00")
    plt.title("残差分布图")
    plt.xlabel("残差（预测值 - 真实值）")
    plt.ylabel("频数")
    plt.tight_layout()
    plt.show()
    print("这张图说明什么：残差越集中在 0 附近越好；若明显偏移或长尾，说明模型在某些区间误差较大。")


def error_by_true_value_bins(frame: pd.DataFrame, n_bins: int = 5) -> pd.DataFrame:
    # 按真实值区间统计误差，帮助判断高值区或低值区谁更难预测。
    if frame.empty or frame[CONFIG["TRUE_COL"]].nunique() < 2:
        return pd.DataFrame()
    temp = frame.copy()
    try:
        temp["真实值区间"] = pd.qcut(temp[CONFIG["TRUE_COL"]], q=min(n_bins, temp[CONFIG["TRUE_COL"]].nunique()), duplicates="drop")
    except Exception:
        return pd.DataFrame()
    return (
        temp.groupby("真实值区间", observed=False)
        .agg(
            样本数=(CONFIG["ID_COL"], "count"),
            平均绝对误差=("abs_error", "mean"),
            RMSE=("residual", lambda x: float(np.sqrt(np.mean(np.square(x))))),
        )
        .reset_index()
    )


uq_df = build_uncertainty_df(uncertainty_payload)
prediction_eval_df, prediction_note = prepare_prediction_df(external_prediction_df, analysis_df, uq_df)
regression_metrics = evaluate_regression(prediction_eval_df)

print(prediction_note)
if prediction_eval_df.empty:
    if not training_metrics_df.empty:
        display(training_metrics_df)
    print("提示：当前没有逐样本预测结果，所以这一节无法回答更细的误差模式问题。")
    regression_analysis = {
        "metrics": {},
        "top_errors": pd.DataFrame(),
        "error_bins": pd.DataFrame(),
        "conclusion_lines": ["请在服务器上提供逐样本预测文件，以启用真实值 vs 预测值、残差图和误差分桶分析。"],
    }
else:
    metrics_df = pd.DataFrame(
        {"指标": list(regression_metrics.keys()), "数值": list(regression_metrics.values())}
    )
    display(metrics_df)
    print("这张表说明什么：它用 MAE、MSE、RMSE、R² 四个角度概括模型整体拟合质量。")

    plot_true_vs_pred(prediction_eval_df)
    plot_residuals(prediction_eval_df)
    plot_residual_distribution(prediction_eval_df)

    top_error_df = prediction_eval_df.sort_values("abs_error", ascending=False).head(CONFIG["TOP_N_ERROR"])
    display(top_error_df)
    print("这张表说明什么：它列出了误差最大的样本，适合回查几何、标注或模型是否在某些结构上持续失效。")

    error_bin_df = error_by_true_value_bins(prediction_eval_df)
    if not error_bin_df.empty:
        display(error_bin_df)
        plt.figure(figsize=(10, 5))
        sns.barplot(data=error_bin_df, x="真实值区间", y="平均绝对误差", color="#56B4E9")
        plt.title("按真实值区间统计的平均绝对误差")
        plt.xlabel("真实值区间")
        plt.ylabel("平均绝对误差")
        plt.xticks(rotation=30, ha="right")
        plt.tight_layout()
        plt.show()
        print("这张图说明什么：它用于判断模型在哪些目标值区间更容易出错。")

    conclusion_lines = [
        f"模型整体 MAE 为 {regression_metrics.get('MAE', float('nan')):.6f}。",
        f"模型整体 RMSE 为 {regression_metrics.get('RMSE', float('nan')):.6f}。",
        f"模型整体 R² 为 {regression_metrics.get('R2', float('nan')):.6f}。",
    ]

    if not error_bin_df.empty:
        worst_row = error_bin_df.sort_values("平均绝对误差", ascending=False).iloc[0]
        best_row = error_bin_df.sort_values("平均绝对误差", ascending=True).iloc[0]
        conclusion_lines.append(f"误差最高的真实值区间是 {worst_row['真实值区间']}。")
        conclusion_lines.append(f"误差最低的真实值区间是 {best_row['真实值区间']}。")

    residual_corr = prediction_eval_df[[CONFIG["PRED_COL"], "residual"]].corr().iloc[0, 1]
    if pd.notna(residual_corr) and abs(residual_corr) >= 0.3:
        conclusion_lines.append("残差与预测值存在一定相关性，说明模型可能有系统性偏差。")
    else:
        conclusion_lines.append("残差与预测值没有表现出特别强的线性趋势，误差模式相对更随机。")

    display(Markdown("### 回归评估结论\n" + "\n".join(f"- {item}" for item in conclusion_lines)))
    regression_analysis = {
        "metrics": regression_metrics,
        "top_errors": top_error_df,
        "error_bins": error_bin_df,
        "conclusion_lines": conclusion_lines,
    }


## 5. 模型为什么这样预测？

这一节分成两个分支：

- `ADL / ANI 主线`：优先分析描述符、误差和不确定性之间的关系，而不是强行套树模型特征重要性
- `树模型 / 表格特征分支`：如果你在服务器上提供了可解释模型，则进一步画 feature importance、permutation importance 和 SHAP

也就是说，这一节不是默认假设“任何模型都有 feature_importances_”，而是根据模型类型走不同路径。


In [ ]:
# 这一个代码块负责解释模型行为，并尽量兼容 ADL 主线与树模型扩展主线。
def choose_descriptor_columns(frame: pd.DataFrame) -> list[str]:
    # 这些字段更像“可解释描述符”，优先拿来观察目标值和误差随它们如何变化。
    preferred = [
        "E_baseline",
        "E_target",
        "delta_E",
        "|F_baseline|",
        "|F_target|",
        "|delta_F|",
        "num_atoms",
        "charge",
        "multiplicity",
        "mean_atomic_number",
        "coordinate_std",
    ]
    available = [column for column in preferred if column in frame.columns and pd.api.types.is_numeric_dtype(frame[column])]
    if available:
        return available[:4]
    numeric_cols, _ = detect_column_groups(frame)
    excluded = {CONFIG["TRUE_COL"], CONFIG["PRED_COL"], "residual", "abs_error"}
    return [column for column in numeric_cols if column not in excluded][:4]


def plot_descriptor_relationships(frame: pd.DataFrame, target_col: str | None) -> None:
    # 观察重要描述符与目标值/误差之间的关系，帮助解释模型在哪些区域更容易出错。
    if frame.empty:
        print("没有可用于解释的数据表，跳过描述符关系图。")
        return

    descriptor_cols = choose_descriptor_columns(frame)
    if not descriptor_cols:
        print("没有合适的数值描述符，跳过描述符关系图。")
        return

    y_column = "abs_error" if "abs_error" in frame.columns else target_col
    if y_column is None or y_column not in frame.columns:
        print("没有可用的目标列或误差列，跳过描述符关系图。")
        return

    n_rows = math.ceil(len(descriptor_cols) / 2)
    fig, axes = plt.subplots(n_rows, 2, figsize=(12, 4.5 * n_rows))
    axes = np.array(axes).reshape(-1)
    for ax, column in zip(axes, descriptor_cols):
        sns.scatterplot(data=frame, x=column, y=y_column, ax=ax, alpha=0.75, s=60)
        ax.set_title(f"{column} 与 {y_column} 的关系")
        ax.set_xlabel(column)
        ax.set_ylabel(y_column)
    for ax in axes[len(descriptor_cols):]:
        ax.axis("off")
    fig.suptitle("重要描述符与目标/误差的关系", y=1.02)
    plt.tight_layout()
    plt.show()
    print("这张图说明什么：它帮助判断模型误差是否集中在某些结构特征或能量区间。")


def plot_uq_vs_error(frame: pd.DataFrame) -> str:
    # 比较不确定性和真实误差是否一致，是主动学习里非常关键的一步。
    required_cols = {"uncertainty", "abs_error"}
    if frame.empty or not required_cols.issubset(frame.columns):
        return "当前没有同时包含 uncertainty 和 abs_error 的样本表，无法判断 UQ 与真实误差是否一致。"

    plt.figure(figsize=(8, 5))
    sns.scatterplot(data=frame, x="uncertainty", y="abs_error", s=60, alpha=0.75)
    plt.title("不确定性 vs 绝对误差")
    plt.xlabel("不确定性（UQ）")
    plt.ylabel("绝对误差")
    plt.tight_layout()
    plt.show()
    print("这张图说明什么：如果 UQ 与绝对误差正相关，说明当前的不确定性估计对主动学习选点是有意义的。")

    corr = frame[["uncertainty", "abs_error"]].corr().iloc[0, 1]
    if pd.isna(corr):
        return "UQ 与绝对误差的相关性无法计算。"
    if corr >= 0.5:
        return f"UQ 与绝对误差呈较明显正相关（相关系数约 {corr:.3f}），说明不确定性估计较有参考价值。"
    if corr >= 0.2:
        return f"UQ 与绝对误差存在一定正相关（相关系数约 {corr:.3f}），但仍建议结合人工检查。"
    return f"UQ 与绝对误差相关性较弱（相关系数约 {corr:.3f}），说明当前 UQ 对真实误差的指示作用有限。"


def compare_high_error_and_high_uq(frame: pd.DataFrame) -> pd.DataFrame:
    # 对比高误差样本和高 UQ 样本的交集，判断主动学习是否真的挑中了困难样本。
    if frame.empty or "abs_error" not in frame.columns or "uncertainty" not in frame.columns:
        return pd.DataFrame()
    error_rank = frame[[CONFIG["ID_COL"], "abs_error"]].sort_values("abs_error", ascending=False).head(CONFIG["TOP_N_ERROR"])
    uq_rank = frame[[CONFIG["ID_COL"], "uncertainty"]].sort_values("uncertainty", ascending=False).head(CONFIG["TOP_N_ERROR"])
    merged = error_rank.merge(uq_rank, on=CONFIG["ID_COL"], how="outer")
    return merged


def get_feature_matrix(frame: pd.DataFrame, target_col: str | None) -> tuple[pd.DataFrame, pd.Series | None]:
    # 为特征重要性和 permutation importance 准备数值特征矩阵。
    if frame.empty or target_col is None or target_col not in frame.columns:
        return pd.DataFrame(), None
    numeric_cols, _ = detect_column_groups(frame)
    feature_cols = [
        column
        for column in numeric_cols
        if column not in {target_col}
        and column not in {CONFIG["TRUE_COL"], CONFIG["PRED_COL"], "residual", "abs_error", "uncertainty"}
    ]
    if not feature_cols:
        return pd.DataFrame(), None
    clean = frame[feature_cols + [target_col]].dropna()
    if clean.empty:
        return pd.DataFrame(), None
    return clean[feature_cols], clean[target_col]


def compute_feature_importance(model: Any, x_frame: pd.DataFrame, y_series: pd.Series | None) -> pd.DataFrame:
    # 对 sklearn 风格模型优先读 feature_importances_，否则退回 permutation importance。
    if model is None or x_frame.empty:
        return pd.DataFrame()

    if hasattr(model, "feature_importances_"):
        return (
            pd.DataFrame({"特征": x_frame.columns, "重要性": model.feature_importances_})
            .sort_values("重要性", ascending=False)
            .reset_index(drop=True)
        )

    if y_series is None:
        return pd.DataFrame()

    try:
        perm = permutation_importance(
            model,
            x_frame,
            y_series,
            n_repeats=10,
            random_state=42,
            scoring="neg_root_mean_squared_error",
        )
        return (
            pd.DataFrame({"特征": x_frame.columns, "重要性": perm.importances_mean})
            .sort_values("重要性", ascending=False)
            .reset_index(drop=True)
        )
    except Exception as exc:  # noqa: BLE001
        print(f"permutation importance 计算失败：{exc}")
        return pd.DataFrame()


def plot_feature_importance(importance_df: pd.DataFrame) -> None:
    # 条形图展示最重要的前若干个特征。
    if importance_df.empty:
        print("没有可展示的特征重要性结果。")
        return
    top_df = importance_df.head(15)
    plt.figure(figsize=(10, max(5, len(top_df) * 0.35)))
    sns.barplot(data=top_df, x="重要性", y="特征", color="#0072B2")
    plt.title("特征重要性条形图")
    plt.xlabel("重要性")
    plt.ylabel("特征")
    plt.tight_layout()
    plt.show()
    print("这张图说明什么：重要性越高，说明该特征对模型输出的影响通常越大。")


def try_shap_summary(model: Any, x_frame: pd.DataFrame) -> str:
    # SHAP 是可选增强项，不可用时只给提示，不让 notebook 崩掉。
    if model is None or x_frame.empty:
        return "缺少模型或特征矩阵，跳过 SHAP 分析。"
    if not module_available("shap"):
        return "当前环境未安装 shap，已跳过 SHAP summary plot。"

    try:
        import shap

        sample_x = x_frame.sample(min(len(x_frame), 300), random_state=42)
        explainer = shap.Explainer(model, sample_x)
        shap_values = explainer(sample_x)
        shap.summary_plot(shap_values, sample_x, show=True)
        return "已完成 SHAP summary plot，可据此查看哪些特征在全局上最影响模型输出。"
    except Exception as exc:  # noqa: BLE001
        return f"SHAP 分析未成功执行：{exc}"


explain_frame = analysis_df.copy()
if not prediction_eval_df.empty and CONFIG["ID_COL"] in explain_frame.columns:
    merge_cols = [CONFIG["ID_COL"], CONFIG["TRUE_COL"], CONFIG["PRED_COL"], "residual", "abs_error"]
    if "uncertainty" in prediction_eval_df.columns:
        merge_cols.append("uncertainty")
    explain_frame = explain_frame.merge(
        prediction_eval_df[merge_cols].drop_duplicates(subset=[CONFIG["ID_COL"]]),
        on=CONFIG["ID_COL"],
        how="left",
    )
elif not prediction_eval_df.empty:
    explain_frame = prediction_eval_df.copy()

plot_descriptor_relationships(explain_frame, effective_target_col)
uq_alignment_message = plot_uq_vs_error(explain_frame)
high_error_uq_df = compare_high_error_and_high_uq(explain_frame)
if not high_error_uq_df.empty:
    display(high_error_uq_df)
    print("这张表说明什么：它用于对比“模型最容易错的样本”和“UQ 最高的样本”是否重合。")

external_model = load_external_model()
feature_matrix, feature_target = get_feature_matrix(explain_frame, effective_target_col)
importance_df = compute_feature_importance(external_model, feature_matrix, feature_target)
if not importance_df.empty:
    display(importance_df)
    plot_feature_importance(importance_df)
elif external_model is None:
    print("未提供外部可解释模型，因此跳过特征重要性分支。")
else:
    print("已经提供外部模型，但当前无法稳定计算特征重要性，请检查模型接口或特征矩阵。")

shap_message = try_shap_summary(external_model, feature_matrix)

explanation_lines = [uq_alignment_message]
if not importance_df.empty:
    top_features = ", ".join(importance_df["特征"].head(5).tolist())
    explanation_lines.append(f"当前最重要的前几个特征大致是：{top_features}。")
else:
    explanation_lines.append("当前默认解释主线仍然是描述符-误差关系与 UQ 行为分析，而不是树模型特征重要性。")
explanation_lines.append(shap_message)

display(Markdown("### 解释性分析结论\n" + "\n".join(f"- {item}" for item in explanation_lines)))
explanation_analysis = {
    "uq_alignment_message": uq_alignment_message,
    "importance_df": importance_df,
    "shap_message": shap_message,
    "high_error_uq_df": high_error_uq_df,
    "conclusion_lines": explanation_lines,
}


## 6. 机器学习实验汇报结论区

这一节会把前面的发现整理成适合实验汇报的中文结论，同时补一张“哪些想法可直接用、哪些只是可选增强”的说明表。

## 7. 复用说明

这份 notebook 设计成模板，因此最后还会给出下一轮或下一套体系的复用提示。


In [ ]:
# 这一个代码块负责输出汇报版总结、适用性说明和复用建议。
experiment_summary_lines = []

if not analysis_df.empty:
    experiment_summary_lines.append(f"当前可分析数据共有 {analysis_df.shape[0]} 条样本、{analysis_df.shape[1]} 个字段。")
else:
    experiment_summary_lines.append("当前仓库没有可直接分析的数据表，建议在服务器环境补充数据后重新运行。")

if isinstance(base_config, dict) and base_config:
    pool_size = base_config.get("sampling", {}).get("default_pool_size")
    initial_points = base_config.get("active_learning", {}).get("initial_points")
    if pool_size is not None:
        experiment_summary_lines.append(f"配置中的几何池大小为 {pool_size}。")
    if initial_points is not None:
        experiment_summary_lines.append(f"配置中的初始训练样本数为 {initial_points}。")

if isinstance(delta_metadata, dict) and delta_metadata.get("num_samples") is not None:
    experiment_summary_lines.append(f"当前 delta 数据集记录的样本数为 {delta_metadata.get('num_samples')}。")

if isinstance(training_diagnostics, dict) and training_diagnostics:
    experiment_summary_lines.append("训练诊断产物已导出到标准路径，可直接供 notebook 读取。")

if isinstance(pipeline_summary, dict) and pipeline_summary:
    experiment_summary_lines.append(f"最近一次主控流程 success = {pipeline_summary.get('success')}。")

if isinstance(train_main_status, dict) and "success" in train_main_status:
    experiment_summary_lines.append(f"主模型训练状态：{'成功' if train_main_status.get('success') else '失败'}。")
if isinstance(train_aux_status, dict) and "success" in train_aux_status:
    experiment_summary_lines.append(f"辅助模型训练状态：{'成功' if train_aux_status.get('success') else '失败'}。")

if history_analysis.get("conclusion_lines"):
    experiment_summary_lines.extend(history_analysis["conclusion_lines"][:3])

if regression_analysis.get("conclusion_lines"):
    experiment_summary_lines.extend(regression_analysis["conclusion_lines"][:3])

if explanation_analysis.get("conclusion_lines"):
    experiment_summary_lines.extend(explanation_analysis["conclusion_lines"][:3])

if isinstance(uncertainty_payload, dict) and uncertainty_payload.get("num_samples") is not None:
    experiment_summary_lines.append(f"本次 UQ 评估覆盖 {uncertainty_payload.get('num_samples')} 个样本。")

if isinstance(selection_summary, dict) and selection_summary:
    selected_count = len(selection_summary.get("selected_sample_ids", []) or selection_summary.get("selected_samples", []))
    uncertain_ratio = selection_summary.get("uncertain_ratio")
    converged = selection_summary.get("converged")
    experiment_summary_lines.append(f"第 1 轮新增样本数为 {selected_count}。")
    if uncertain_ratio is not None:
        experiment_summary_lines.append(f"当前高不确定性比例约为 {uncertain_ratio:.2%}。")
    if converged is not None:
        experiment_summary_lines.append(f"当前是否满足主动学习收敛判据：{bool(converged)}。")

display(Markdown("### 面向实验汇报的自动结论\n" + "\n".join(f"- {item}" for item in experiment_summary_lines)))

applicability_df = pd.DataFrame(
    [
        {"类别": "可直接用", "内容": "EDA 全套", "说明": "包括缺失值、异常值、偏态、相关性、目标分布。"},
        {"类别": "可直接用", "内容": "回归指标 RMSE / MAE / R²", "说明": "前提是提供逐样本真实值和预测值。"},
        {"类别": "可直接用", "内容": "真实值 vs 预测值、残差图、残差分布图", "说明": "默认作为回归主路径。"},
        {"类别": "可直接用", "内容": "history 自动结论", "说明": "前提是提供逐 epoch 训练记录。"},
        {"类别": "可选用", "内容": "accuracy 曲线", "说明": "只有 history 中存在 accuracy 字段时才绘制。"},
        {"类别": "可选用", "内容": "feature importance / permutation importance", "说明": "适合 sklearn 风格表格模型。"},
        {"类别": "可选用", "内容": "SHAP", "说明": "当前环境装有 shap 且模型兼容时再启用。"},
        {"类别": "这次不作为主路径", "内容": "分类专用指标与图", "说明": "当前 notebook 默认围绕回归任务组织。"},
        {"类别": "这次不作为主路径", "内容": "对 ANI 主模型直接套 feature_importances_", "说明": "当前 ADL 主线更适合看描述符、误差和 UQ 的关系。"},
    ]
)
display(applicability_df)
print("这张表说明什么：它明确区分了当前模板的主线能力、可选增强项和不建议强行套用的分析。")

reuse_df = pd.DataFrame(
    [
        {"优先修改项": "FEATURE_TABLE_PATH", "什么时候改": "服务器上有额外样本级特征表时", "说明": "优先提升 EDA 和解释性分析质量。"},
        {"优先修改项": "HISTORY_PATH", "什么时候改": "想画训练/验证 loss 曲线时", "说明": "标准路径下默认读取 train_main_history.json；若 history 缺失会自动降级。"},
        {"优先修改项": "PREDICTION_PATH", "什么时候改": "想做真实值 vs 预测值和残差分析时", "说明": "标准路径下默认读取 train_main_predictions.csv；若切到辅助模型或自定义结果，再改这里。"},
        {"优先修改项": "MODEL_VIEW", "什么时候改": "想一键切换主模型/辅助模型分析时", "说明": "切到 aux 后，history 和 prediction 会自动改读辅助模型产物。"},
        {"优先修改项": "MODEL_PATH", "什么时候改": "需要做 feature importance 或 SHAP 时", "说明": "适合 sklearn 树模型或兼容解释器的模型。"},
        {"优先修改项": "TARGET_COL", "什么时候改": "目标不是 delta_E 时", "说明": "例如你要分析别的回归目标。"},
        {"优先修改项": "STANDARD_PATHS", "什么时候改": "跑第二轮或更换体系时", "说明": "如果目录结构变了，需要同步调整约定路径。"},
    ]
)
display(reuse_df)
print("这张表说明什么：以后跑第二轮或换体系时，通常优先改这里列出来的几个入口，不必重写整份 notebook。")
